# 02. Double DQN Training on CartPole-v1

This notebook guides you through:
1. Loading standard hyperparameters from `config/dqn.yaml`.
2. Training Double DQN with Experience Replay and Soft Target Updates.
3. Visualizing training learning curves and exploration schedules.
4. Evaluating the best checkpoint on independent test seeds (100-199).

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

import argparse
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from libraries.dqn import train, DQNAgent
from libraries.policies import get_policy
from libraries.evaluate import evaluate_policy

## 1. Load Double DQN Hyperparameters

In [ ]:
with open('config/dqn.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
print('Default configuration:')
for k, v in cfg.items():
    print(f'  {k:20s}: {v}')

## 2. Train Agent (Seed 0)

You can reduce `total_steps` to 10000 for a quick smoke test.

In [ ]:
args = argparse.Namespace(**cfg)
args.seed = 0
args.tag = 'demo_nb'
args.total_steps = 40000
args.eval_every = 2500
args.eval_episodes = 10
train(args)

## 3. Visualize Training Progression

Read CSV training logs from `private/logs/dqn/` and plot moving average returns.

In [ ]:
df = pd.read_csv('private/logs/dqn/demo_nb_seed0.csv')
df['ma_return'] = df['return'].rolling(window=20, min_periods=1).mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
ax1.plot(df['env_step'], df['return'], alpha=0.3, color='steelblue', label='Episode return')
ax1.plot(df['env_step'], df['ma_return'], color='darkblue', linewidth=2, label='Moving avg (20)')
ax1.axhline(475, color='green', linestyle='--', label='Solved threshold (475)')
ax1.set_title('Learning Curve - Double DQN')
ax1.set_xlabel('Environment Steps')
ax1.set_ylabel('Return')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(df['env_step'], df['epsilon'], color='orange', label='Epsilon decay')
ax2.set_title('Epsilon Exploration Schedule')
ax2.set_xlabel('Environment Steps')
ax2.set_ylabel('Epsilon')
ax2.grid(True, alpha=0.3)
ax2.legend()
plt.tight_layout()
plt.show()

## 4. Evaluate Best Checkpoint on Test Seeds (100-199)

In [ ]:
model_path = 'models/dqn_demo_nb_seed0_best.pt'
policy = get_policy('dqn', model_path)
eval_res = evaluate_policy(policy, n_episodes=100, seed_start=100)
print(f'Post-training mean return: {eval_res["mean"]:.2f} +- {eval_res["std"]:.2f}')
print(f'Fraction reaching max 500 score: {eval_res["fraction_500"] * 100:.1f}%')